# Active-Set Method for Convex Quadratic Programming

Solve a two-dimensional strictly convex quadratic program with linear inequality constraints using a working-set KKT system. Compare an interior starting point, a vertex and a non-vertex boundary point.

[Technical notes](../docs/active_set_quadratic_programming.md) · [Saved figures](../results/active_set_quadratic_programming/)

In [1]:
import numpy as np

## Objective and feasible region

$$\min\; x_1^2+2x_2^2-2x_1-6x_2-2x_1x_2$$

subject to $x_1+x_2\le2$, $-x_1+2x_2\le2$ and $x_1,x_2\ge0$.

## Quadratic form

Write $f(x)=\tfrac12x^THx+c^Tx$ with $H=\begin{bmatrix}2&-2\\-2&4\end{bmatrix}$ and $c=(-2,-6)^T$. The leading principal minors are 2 and 4, so $H$ is positive definite.

In [2]:
H = np.array([[2.0, -2.0],
              [-2.0, 4.0]])
c = np.array([-2.0, -6.0])


## Matrix form of inequality constraints

Encode the four inequalities as $Ax\le b$ in the order shown by the code. Python uses zero-based indices.

In [3]:
A = np.array([[1.0, 1.0],    # x1 + x2 <= 2
              [-1.0, 2.0],   # -x1 + 2x2 <= 2
              [-1.0, 0.0],   # x1 >= 0
              [0.0, -1.0]])  # x2 >= 0
b = np.array([2.0, 2.0, 0.0, 0.0])


## Working-set KKT solver

Solve $\begin{bmatrix}H&A_W^T\\A_W&0\end{bmatrix}\begin{bmatrix}p\\\lambda\end{bmatrix}=\begin{bmatrix}-g\\0\end{bmatrix}$. If $p=0$ and multipliers are nonnegative, stop; otherwise drop the most negative multiplier. For a nonzero direction, choose a feasible step limited by blocking inequalities and refresh the working set.

In [11]:
def f(x):
    return 0.5 * x @ H @ x + c @ x

def grad(x):
    return H @ x + c

def feasible(x, tol=1e-10):
    return np.all(b - A @ x >= -tol)

def active_set(x, tol=1e-10):
    return [i for i in range(len(b)) if abs(b[i] - A[i] @ x) <= tol]

def solve_kkt(H, g, A_w):
    if A_w.size == 0:
        return np.linalg.solve(H, -g), np.array([])

    KKT = np.block([
        [H, A_w.T],
        [A_w, np.zeros((A_w.shape[0], A_w.shape[0]))]
    ])
    rhs = np.concatenate([-g, np.zeros(A_w.shape[0])])
    sol = np.linalg.solve(KKT, rhs)
    return sol[:2], sol[2:]

def active_set_qp(x0, max_iter=100, tol=1e-10):
    if not feasible(x0):
        raise ValueError("Initial point is not feasible.")

    x = x0.copy()
    W = active_set(x)

    for k in range(max_iter):
        g = grad(x)
        A_w = A[W, :] if len(W) > 0 else np.zeros((0, 2))
        p, lam = solve_kkt(H, g, A_w)

        if np.linalg.norm(p) <= tol:
            if len(W) == 0 or np.min(lam) >= -tol:
                return x, f(x), k, W
            W.pop(int(np.argmin(lam)))
            continue

        alpha = 1.0
        blocking = None
        for i in range(len(b)):
            if i in W:
                continue
            denom = A[i] @ p
            if denom > tol:
                step = (b[i] - A[i] @ x) / denom
                if step < alpha:
                    alpha = step
                    blocking = i

        x = x + alpha * p

        if blocking is not None and blocking not in W:
            W.append(blocking)

        W = active_set(x)

    return x, f(x), max_iter, W


## Compare three feasible initial points

Start at $(0.5,0.3)$, $(0,0)$ and $(1.5,0.5)$.

In [12]:
starts = {
    "Interior": np.array([0.5, 0.3]),
    "Vertex": np.array([0.0, 0.0]),
    "Boundary (non-vertex)": np.array([1.5, 0.5])
}

for name, x0 in starts.items():
    x_star, f_star, iters, W_star = active_set_qp(x0)
    print(f"\n{name} start")
    print("x* =", x_star)
    print("f(x*) =", f_star)
    print("iterations =", iters)
    print("Active constraints W* =", [i+1 for i in W_star])


Interior start
x* = [0.8 1.2]
f(x*) = -7.199999999999999
iterations = 2
Active constraints W* = [1]

Vertex start
x* = [0.8 1.2]
f(x*) = -7.199999999999999
iterations = 6
Active constraints W* = [1]

Boundary (non-vertex) start
x* = [0.8 1.2]
f(x*) = -7.2
iterations = 1
Active constraints W* = [1]


## Results and interpretation

| Starting point | Reported solution | Objective | Iteration index |
|---|---|---:|---:|
| Interior | (0.8, 1.2) | -7.2 | 2 |
| Vertex | (0.8, 1.2) | -7.2 | 6 |
| Non-vertex boundary | (0.8, 1.2) | -7.2 | 1 |

The numerical solution is the unique optimum. Indeed, at $(0.8,1.2)$ the gradient is $(-2.8,-2.8)$ and the first constraint is tight. Its multiplier is 2.8 in the scaled representation $x_1+x_2\le2$ (or 5.6 for the original half-scaled constraint). The other inequalities are slack, so their multipliers are zero.

**Output consistency note:** the saved output prints `W* = [1]` in all three cases, but for the displayed constraint matrix the tight row is index **0**, not 1: $-0.8+2(1.2)=1.6<2$. The original printed indices are retained; the KKT verification above is the mathematically consistent interpretation. The experiment shows that the starting point influences the number of working-set transitions, but all displayed solutions agree.